**Tahap 1A: Memuat Dataset Wikipedia ID di Colab**

In [1]:
# Install pustaka datasets jika belum ada di environment Colab
!pip install -q datasets

from datasets import load_dataset

# 1. Memuat 2.000 artikel pertama dari subset Wikipedia Bahasa Indonesia
print("Memuat dataset Wikipedia Bahasa Indonesia (2.000 artikel)...")
raw_dataset = load_dataset("wikimedia/wikipedia", "20231101.id", split="train[:2000]")

# 2. Mengekstrak teks dari tiap artikel ke dalam list
texts = [item['text'] for item in raw_dataset]

# 3. Verifikasi hasil pemuatan
print(f"Berhasil memuat {len(texts)} artikel.")
print("\n--- Contoh Potongan Artikel Pertama ---")
print(texts[0][:300] + "...")

Memuat dataset Wikipedia Bahasa Indonesia (2.000 artikel)...


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

20231101.id/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  267MB            

20231101.id/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

20231101.id/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  146MB            

20231101.id/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

20231101.id/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  170MB            

20231101.id/train-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/665622 [00:00<?, ? examples/s]

Berhasil memuat 2000 artikel.

--- Contoh Potongan Artikel Pertama ---
Asam deoksiribonukleat, lebih dikenal dengan singkatan DNA (bahasa Inggris: deoxyribonucleic acid), adalah salah satu jenis asam nukleat yang memiliki kemampuan pewarisan sifat. Keberadaan asam deoksiribonukleat ditemukan di dalam nukleoprotein yang membentuk inti sel. James Dewey Watson dan Francis...


In [2]:
from datasets import load_dataset

print("Memuat dataset Wikipedia Bahasa Indonesia (Streaming 2.000 artikel)...")

# 1. Gunakan streaming=True agar tidak mengunduh seluruh file Parquet
streamed_dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.id",
    split="train",
    streaming=True
)

# 2. Ambil tepat 2.000 artikel pertama saja
texts = []
for item in streamed_dataset.take(2000):
    texts.append(item['text'])

# 3. Verifikasi hasil pemuatan
print(f"Berhasil memuat {len(texts)} artikel secara instan.")
print("\n--- Contoh Potongan Artikel Pertama ---")
print(texts[0][:300] + "...")

Memuat dataset Wikipedia Bahasa Indonesia (Streaming 2.000 artikel)...
Berhasil memuat 2000 artikel secara instan.

--- Contoh Potongan Artikel Pertama ---
Asam deoksiribonukleat, lebih dikenal dengan singkatan DNA (bahasa Inggris: deoxyribonucleic acid), adalah salah satu jenis asam nukleat yang memiliki kemampuan pewarisan sifat. Keberadaan asam deoksiribonukleat ditemukan di dalam nukleoprotein yang membentuk inti sel. James Dewey Watson dan Francis...


**Pembersihan Teks (Text Preprocessing)**

In [3]:
import re

def clean_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Hapus karakter selain huruf a-z dan spasi
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 3. Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Uji coba pembersihan pada 2.000 artikel
print("Membersihkan 2.000 artikel Wikipedia...")
cleaned_texts = [clean_text(doc) for doc in texts]

# Verifikasi hasil pembersihan artikel pertama
print("\n--- Hasil Pembersihan Artikel Pertama (200 Karakter Pertama) ---")
print(cleaned_texts[0][:200] + "...")

Membersihkan 2.000 artikel Wikipedia...

--- Hasil Pembersihan Artikel Pertama (200 Karakter Pertama) ---
asam deoksiribonukleat lebih dikenal dengan singkatan dna bahasa inggris deoxyribonucleic acid adalah salah satu jenis asam nukleat yang memiliki kemampuan pewarisan sifat keberadaan asam deoksiribonu...


**Pembuatan Kosakata (Vocabulary)**

In [4]:
from collections import Counter

# 1. Hitung frekuensi seluruh kata dari 2.000 artikel
all_words = []
for doc in cleaned_texts:
    all_words.extend(doc.split())

word_counts = Counter(all_words)

# 2. Definisikan Token Khusus BERT
special_tokens = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']

# 3. Ambil 7.995 kata terbanyak agar total vocab tepat 8.000
vocab_size_target = 8000
num_top_words = vocab_size_target - len(special_tokens)
most_common_words = [word for word, count in word_counts.most_common(num_top_words)]

# 4. Gabungkan special tokens dan top words menjadi daftar vocab utuh
vocab = special_tokens + most_common_words

# 5. Buat dictionary pemetaan (Word to Index & Index to Word)
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for idx, word in enumerate(vocab)}

# Verifikasi hasil pemodelan Vocab
print(f"Total kata unik ditemukan di corpus: {len(word_counts)}")
print(f"Ukuran Vocabulary final yang digunakan: {len(vocab)}")
print("\n--- 10 Kata/Token Pertama di Vocab (termasuk Special Tokens) ---")
print(vocab[:10])

Total kata unik ditemukan di corpus: 115584
Ukuran Vocabulary final yang digunakan: 8000

--- 10 Kata/Token Pertama di Vocab (termasuk Special Tokens) ---
['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', 'dan', 'yang', 'di', 'dari', 'pada']


**Fungsi Konversi Teks ke Token ID (Encode & Decode)**

In [5]:
def encode(text, word2idx_map):
    # 1. Pecah teks bersih menjadi kata-kata
    words = clean_text(text).split()
    # 2. Ubah kata ke ID, gunakan ID [UNK] (1) jika kata tidak ada di vocab
    ids = [word2idx_map.get(word, word2idx_map['[UNK]']) for word in words]
    return ids

def decode(ids, idx2word_map):
    # Ubah ID kembali ke kata
    words = [idx2word_map.get(idx, '[UNK]') for idx in ids]
    return " ".join(words)

# Uji coba fungsi encode dan decode pada contoh kalimat
sample_sentence = "DNA adalah asam nukleat yang menyimpan informasi genetik rahasia."
encoded_ids = encode(sample_sentence, word2idx)
decoded_text = decode(encoded_ids, idx2word)

print("--- Pengujian Fungsi Encode & Decode ---")
print(f"Kalimat Asli    : {sample_sentence}")
print(f"Encoded (IDs)   : {encoded_ids}")
print(f"Decoded (Teks)  : {decoded_text}")

--- Pengujian Fungsi Encode & Decode ---
Kalimat Asli    : DNA adalah asam nukleat yang menyimpan informasi genetik rahasia.
Encoded (IDs)   : [1492, 15, 1331, 1, 6, 2674, 436, 3315, 3092]
Decoded (Teks)  : dna adalah asam [UNK] yang menyimpan informasi genetik rahasia


**Membuat BERTDataset Dasar**

In [6]:
import torch
from torch.utils.data import Dataset

class BERTDataset(Dataset):
    def __init__(self, texts, word2idx, max_len=128):
        self.texts = texts
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # 1. Encode teks ke ID kata dasar menggunakan fungsi encode dari Langkah 1
        raw_ids = encode(text, self.word2idx)

        # 2. Potong teks jika terlalu panjang (sisakan 2 ruang untuk [CLS] dan [SEP])
        max_words = self.max_len - 2
        if len(raw_ids) > max_words:
            raw_ids = raw_ids[:max_words]

        # 3. Tambahkan token [CLS] di awal dan [SEP] di akhir
        input_ids = [self.word2idx['[CLS]']] + raw_ids + [self.word2idx['[SEP]']]

        # 4. Buat attention_mask awal (sebelum padding)
        attention_mask = [1] * len(input_ids)

        # 5. Lakukan Padding jika panjang sekuens kurang dari max_len (128)
        pad_len = self.max_len - len(input_ids)
        if pad_len > 0:
            input_ids += [self.word2idx['[PAD]']] * pad_len
            attention_mask += [0] * pad_len

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long)
        }

# --- Uji Coba Dataset Dasar ---
print("Membuat instance BERTDataset...")
dataset_test = BERTDataset(cleaned_texts, word2idx, max_len=128)

# Ambil sampel artikel pertama (indeks 0)
sample = dataset_test[0]

print("\n--- Hasil Verifikasi Sub-langkah 2A ---")
print(f"Shape input_ids      : {sample['input_ids'].shape}")
print(f"Shape attention_mask : {sample['attention_mask'].shape}")
print(f"ID Token Pertama ([CLS]) : {sample['input_ids'][0].item()} (Harus ID [CLS] = 2)")
print(f"ID Token Terakhir di sekuens: {sample['input_ids'][-1].item()} (Jika ada padding = 0)")

Membuat instance BERTDataset...

--- Hasil Verifikasi Sub-langkah 2A ---
Shape input_ids      : torch.Size([128])
Shape attention_mask : torch.Size([128])
ID Token Pertama ([CLS]) : 2 (Harus ID [CLS] = 2)
ID Token Terakhir di sekuens: 3 (Jika ada padding = 0)


**Logika MLM Masking & DataLoader**

In [7]:
import torch
import random
from torch.utils.data import Dataset, DataLoader

class BERTMaskDataset(Dataset):
    def __init__(self, texts, word2idx, max_len=128, mask_prob=0.15):
        self.texts = texts
        self.word2idx = word2idx
        self.max_len = max_len
        self.mask_prob = mask_prob

        # Simpan daftar ID token khusus agar tidak ikut ter-mask
        self.special_token_ids = {
            word2idx['[PAD]'],
            word2idx['[UNK]'],
            word2idx['[CLS]'],
            word2idx['[SEP]'],
            word2idx['[MASK]']
        }

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        raw_ids = encode(text, self.word2idx)

        max_words = self.max_len - 2
        if len(raw_ids) > max_words:
            raw_ids = raw_ids[:max_words]

        input_ids = [self.word2idx['[CLS]']] + raw_ids + [self.word2idx['[SEP]']]
        attention_mask = [1] * len(input_ids)

        pad_len = self.max_len - len(input_ids)
        if pad_len > 0:
            input_ids += [self.word2idx['[PAD]']] * pad_len
            attention_mask += [0] * pad_len

        # --- LOGIKA MASKING MLM (15%) ---
        labels = [-100] * self.max_len  # Inisialisasi label dengan -100
        masked_input_ids = input_ids.copy()

        for i in range(len(input_ids)):
            token_id = input_ids[i]

            # Jangan lakukan masking jika token adalah special token (termasuk [PAD])
            if token_id in self.special_token_ids:
                continue

            # Peluang 15% untuk memilih token sebagai target prediksi
            if random.random() < self.mask_prob:
                labels[i] = token_id  # Simpan jawaban asli di tensor labels

                prob = random.random()
                if prob < 0.8:
                    # 80% ganti ke [MASK]
                    masked_input_ids[i] = self.word2idx['[MASK]']
                elif prob < 0.9:
                    # 10% ganti ke kata acak dalam vocab (ID 5 ke atas)
                    random_word_id = random.randint(5, len(self.word2idx) - 1)
                    masked_input_ids[i] = random_word_id
                # 10% sisanya tetap tidak diubah (masked_input_ids[i] = token_id)

        return {
            'input_ids': torch.tensor(masked_input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long)
        }

# --- Membuat DataLoader & Verification ---
print("Membuat BERTMaskDataset & DataLoader...")
train_dataset = BERTMaskDataset(cleaned_texts, word2idx, max_len=128)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Ambil 1 batch data untuk dites
first_batch = next(iter(train_loader))

print("\n--- Hasil Verifikasi Sub-langkah 2B ---")
print(f"Batch input_ids shape      : {first_batch['input_ids'].shape}")
print(f"Batch attention_mask shape : {first_batch['attention_mask'].shape}")
print(f"Batch labels shape         : {first_batch['labels'].shape}")

# Hitung berapa banyak token yang di-mask pada sampel pertama di batch
sample_labels = first_batch['labels'][0]
num_masked = (sample_labels != -100).sum().item()
print(f"Jumlah token di-mask di sampel pertama: {num_masked} token")

Membuat BERTMaskDataset & DataLoader...

--- Hasil Verifikasi Sub-langkah 2B ---
Batch input_ids shape      : torch.Size([16, 128])
Batch attention_mask shape : torch.Size([16, 128])
Batch labels shape         : torch.Size([16, 128])
Jumlah token di-mask di sampel pertama: 11 token


**Membangun Modul BERTEmbedding**

In [8]:
import torch
import torch.nn as nn

class BERTEmbedding(nn.Module):
    def __init__(self, vocab_size=8000, embed_size=128, max_len=128, dropout=0.1):
        super().__init__()
        # 1. Token Embedding: mengubah ID kata ke vektor d_model (128)
        self.token_embed = nn.Embedding(vocab_size, embed_size, padding_idx=0)

        # 2. Positional Embedding: mempelajari posisi token 0 s.d. max_len-1
        self.pos_embed = nn.Embedding(max_len, embed_size)

        self.dropout = nn.Dropout(p=dropout)
        self.layer_norm = nn.LayerNorm(embed_size)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        # Buat tensor posisi [0, 1, 2, ..., seq_len-1] untuk setiap sampel di batch
        pos_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).repeat(batch_size, 1)

        # Penjumlahan Token Embedding + Positional Embedding
        embedding = self.token_embed(input_ids) + self.pos_embed(pos_ids)

        return self.dropout(self.layer_norm(embedding))

# --- Uji Coba Modul BERTEmbedding dengan Batch dari Langkah 2 ---
embedding_layer = BERTEmbedding(vocab_size=8000, embed_size=128, max_len=128)

# Ambil input_ids dari first_batch Langkah 2B
sample_input_ids = first_batch['input_ids']
embedded_output = embedding_layer(sample_input_ids)

print("--- Hasil Verifikasi Sub-langkah 3A ---")
print(f"Input Shape  (IDs)       : {sample_input_ids.shape}")
print(f"Output Shape (Embedding) : {embedded_output.shape}")

--- Hasil Verifikasi Sub-langkah 3A ---
Input Shape  (IDs)       : torch.Size([16, 128])
Output Shape (Embedding) : torch.Size([16, 128, 128])


**Modul MultiHeadAttention**

In [9]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size=128, num_heads=4, dropout=0.1):
        super().__init__()
        self.embed_size = embed_size
        self.num_heads = num_heads
        self.head_dim = embed_size // num_heads

        assert self.head_dim * num_heads == embed_size, "embed_size harus habis dibagi num_heads"

        # Proyeksi Linear untuk Q, K, V dan Output
        self.q_linear = nn.Linear(embed_size, embed_size)
        self.k_linear = nn.Linear(embed_size, embed_size)
        self.v_linear = nn.Linear(embed_size, embed_size)
        self.out_linear = nn.Linear(embed_size, embed_size)

        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        batch_size = q.shape[0]

        # 1. Proyeksi Linear Q, K, V & Split ke Multi-Head -> Shape: [batch_size, num_heads, seq_len, head_dim]
        Q = self.q_linear(q).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(k).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(v).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. Scaled Dot-Product Attention -> Shape: [batch_size, num_heads, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 3. Terapkan attention_mask jika ada (ubah nilai mask [PAD] 0 menjadi -1e9)
        if mask is not None:
            # Ubah shape mask agar cocok dengan scores: [batch_size, 1, 1, seq_len]
            extended_mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(extended_mask == 0, -1e9)

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 4. Kalikan dengan V -> Shape: [batch_size, num_heads, seq_len, head_dim]
        context = torch.matmul(attn_weights, V)

        # 5. Concat kembali seluruh head -> Shape: [batch_size, seq_len, embed_size]
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.embed_size)

        # 6. Proyeksi Linear Akhir
        return self.out_linear(context)

# --- Uji Coba Modul MultiHeadAttention ---
mha_layer = MultiHeadAttention(embed_size=128, num_heads=4)

# Ambil mask dari first_batch Langkah 2B
sample_mask = first_batch['attention_mask']

# Masukkan output embedding dari Sub-langkah 3A ke MHA
mha_output = mha_layer(embedded_output, embedded_output, embedded_output, mask=sample_mask)

print("--- Hasil Verifikasi Sub-langkah 3B ---")
print(f"Input Mask Shape   : {sample_mask.shape}")
print(f"Output MHA Shape   : {mha_output.shape}")

--- Hasil Verifikasi Sub-langkah 3B ---
Input Mask Shape   : torch.Size([16, 128])
Output MHA Shape   : torch.Size([16, 128, 128])


**TransformerBlock & Model MiniBERT Utuh**

In [10]:
import torch
import torch.nn as nn

# 1. Position-wise Feed-Forward Network
class PositionwiseFeedForward(nn.Module):
    def __init__(self, embed_size=128, d_ff=512, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(embed_size, d_ff)
        self.fc2 = nn.Linear(d_ff, embed_size)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(self.gelu(self.fc1(x))))

# 2. Transformer Block (Encoder Layer)
class TransformerBlock(nn.Module):
    def __init__(self, embed_size=128, num_heads=4, d_ff=512, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(embed_size, num_heads, dropout)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.feed_forward = PositionwiseFeedForward(embed_size, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Sublayer 1: Self-Attention + Residual & LayerNorm
        attn_out = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Sublayer 2: Feed-Forward + Residual & LayerNorm
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

# 3. Model MiniBERT Utuh (untuk MLM)
class MiniBERT(nn.Module):
    def __init__(self, vocab_size=8000, embed_size=128, num_layers=2, num_heads=4, d_ff=512, max_len=128, dropout=0.1):
        super().__init__()
        self.embedding = BERTEmbedding(vocab_size, embed_size, max_len, dropout)

        # Stack 2 Transformer Encoder Blocks
        self.encoder_layers = nn.ModuleList([
            TransformerBlock(embed_size, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # MLM Head: memproyeksikan fitur 128-dim ke 8.000 vocab (skor logit tiap kata)
        self.mlm_head = nn.Linear(embed_size, vocab_size)

    def forward(self, input_ids, attention_mask):
        # Pass ke Embedding
        x = self.embedding(input_ids)

        # Pass ke 2 Transformer Encoder Layers
        for layer in self.encoder_layers:
            x = layer(x, attention_mask)

        # Logits untuk prediksi MLM -> Shape: [batch_size, seq_len, vocab_size]
        logits = self.mlm_head(x)
        return logits

# --- Uji Coba Model MiniBERT Utuh ---
print("Membuat instance model MiniBERT...")
model = MiniBERT(vocab_size=8000, embed_size=128, num_layers=2, num_heads=4, d_ff=512)

# Tes forwarding menggunakan batch dari Langkah 2B
sample_input_ids = first_batch['input_ids']
sample_mask = first_batch['attention_mask']

output_logits = model(sample_input_ids, sample_mask)

print("\n--- Hasil Verifikasi Langkah 3 (Selesai) ---")
print(f"Input Shape (Batch Input IDs) : {sample_input_ids.shape}")
print(f"Output Logits Shape           : {output_logits.shape}")

Membuat instance model MiniBERT...

--- Hasil Verifikasi Langkah 3 (Selesai) ---
Input Shape (Batch Input IDs) : torch.Size([16, 128])
Output Logits Shape           : torch.Size([16, 128, 8000])
